# GPT-2 파인튜닝

| 항목                     | `distilbert-base-uncased` | `gpt2`                    |
| ---------------------- | ------------------------- | ------------------------- |
| 🔢 모델 타입               | Encoder (BERT 계열)         | Decoder (GPT 계열)          |
| 📦 사전학습 목적             | 마스킹된 언어 모델(Masked LM)     | 오토리그레시브 언어 생성(Next token) |
| 🧠 파라미터 수              | 약 66M                     | 약 124M                    |
| 🏗️ 레이어 수              | 6                         | 12                        |
| 🧩 히든 크기 (Hidden Size) | 768                       | 768                       |
| 🔁 어텐션 헤드 수            | 12                        | 12                        |
| 📏 최대 시퀀스 길이           | 512 tokens                | 1024 tokens               |
| 📄 토크나이저 타입            | WordPiece (BERT형)         | Byte-Pair Encoding (BPE)  |
| 💬 생성 능력               | ❌ (비생성형, 분류에 적합)          | ✅ (텍스트 생성에 최적)            |
| ⚙️ 사용 예시               | 텍스트 분류, 감정 분석 등           | 텍스트 생성, 요약, 질문응답 등        |
| ⚡ 연산 효율성               | 빠름 (BERT의 경량화 버전)         | 느림 (텍스트 생성 연속 수행 필요)      |



# 🧠 GPT-2 파인튜닝, 전체 흐름 한눈에 보기

## 🎯 목표
GPT-2에게 **우리만의 질문-답변 방식**을 가르쳐서, 원하는 스타일로 대답하게 만드는 것!

## 🛠️ 사용하는 기술
- **GPT-2**: 텍스트 생성에 특화된 언어 모델
- **LoRA**: 모델 전체를 바꾸지 않고 일부만 효율적으로 조정하는 기술
- **Trainer**: 학습을 쉽게 해주는 자동화 도구

## 📋 전체 단계 요약

1. **도구 설치**: 필요한 라이브러리 불러오기
2. **모델 로드**: GPT-2와 토크나이저 불러오기
3. **LoRA 설정**: 빠르고 가볍게 학습할 수 있도록 구성
4. **데이터 준비**: 질문-답변 예시 입력
5. **토큰화**: 텍스트 → 숫자 변환
6. **학습 설정**: 반복 횟수, 배치 크기 등 지정
7. **학습 실행**: 모델에게 예시 보여주며 훈련
8. **결과 확인**: 새 질문 넣어보기

## 💡 핵심 요약
- GPT-2는 원래 똑똑함 → 우리는 "내 스타일(내데이터)"로 조금만 바꿈
- LoRA 덕분에 가볍고 빠르게 가능



In [1]:
# STEP 1: 환경 설정
# uv add transformers datasets peft accelerate bitsandbytes

In [2]:
# ✅ STEP 2: 기본 라이브러리 임포트
# 모델 학습에 필요한 필수 라이브러리들을 불러옵니다.
import torch  # PyTorch: 딥러닝 프레임워크
from datasets import Dataset  # 텍스트 데이터를 Dataset 객체로 변환하는 데 사용
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import get_peft_model, LoraConfig, TaskType  # LoRA 기반 파인튜닝 지원

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [3]:
from huggingface_hub import login
from dotenv import load_dotenv
import os

# .env 파일 로드
load_dotenv(override=True)

HF_TOKEN = os.getenv("HF_TOKEN")
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
# ✅ STEP 3: 모델 및 토크나이저 로드
# Hugging Face에서 제공하는 GPT-2 모델과 토크나이저를 불러옵니다.

model_id = "gpt2"  # 사용할 모델 지정 (GPT-2)

# 토크나이저는 텍스트를 숫자 시퀀스로 바꿔주는 도구입니다.
tokenizer = AutoTokenizer.from_pretrained(model_id)

# GPT-2는 pad_token이 기본적으로 없으므로, eos_token(문장 종료 토큰)을 pad_token으로 설정합니다.
# eos_token은 문장의 끝을 모델에게 알려주어 텍스트 생성을 멈추게 함
tokenizer.pad_token = tokenizer.eos_token

# 사전학습된 GPT-2 모델을 불러옵니다. 텍스트 생성 작업에 최적화된 모델입니다.
model = AutoModelForCausalLM.from_pretrained(model_id)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [5]:
# ✅ STEP 4: LoRA 설정
# [포인트] LoRA(Low-Rank Adaptation)는 대규모 모델을 가볍게 미세조정할 수 있게 해주는 기술입니다.
# 전체 모델을 학습시키지 않고, 일부 작은 파라미터만 추가로 학습하기 때문에 훨씬 효율적입니다.

lora_config = LoraConfig(
    r=16,  # 랭크: LoRA 내부 차원. 작을수록 가볍고 빠름
    lora_alpha=8,  # 학습 안정성을 위한 스케일링 계수
    target_modules=["c_attn","c_proj","q_attn"],  # GPT-2에서 LoRA를 적용할 레이어. 'c_attn'은 Attention 모듈의 핵심 부분입니다.
    # 🔍 GPT-2의 레이어들
    # - "c_attn": 쿼리, 키, 밸류를 생성하는 핵심 어텐션 입력 레이어
    # - "c_proj": 어텐션 출력 벡터를 변환하는 투영 레이어
    # - "q_attn": self-attention 중 쿼리 연산에 관여하는 부분 (GPT-J 등 일부 구조에 존재)
    # - "mlp.c_fc": 피드포워드 네트워크의 첫 번째 선형 계층 (MLP의 입력 부분)
    # - "mlp.c_proj": MLP의 출력 부분
    lora_dropout=0.05,  # 드롭아웃 적용 (과적합 방지)
    bias="none",  # bias 파라미터는 학습하지 않음
    task_type=TaskType.CAUSAL_LM,  # 작업 유형: 언어 생성 (Causal Language Modeling)
)

# ✅ 모델에 LoRA 설정을 적용합니다.
# 기존 GPT-2 위에 LoRA 구조를 얹어, 일부 파라미터만 학습 가능하게 만듭니다.
# 추론 시에는 원래 모델과 LoRA가 함께 사용되므로 merge 없이도 작동합니다.
model = get_peft_model(model, lora_config)

# 실제로 학습 가능한 파라미터 수를 출력해봅니다 (LoRA 파라미터만 학습되므로 매우 적습니다).
model.print_trainable_parameters()

trainable params: 1,622,016 || all params: 126,061,824 || trainable%: 1.2867


/mnt/e/gg_ai_merbership_1th/qlora_ft_ex/.venv/lib/python3.12/site-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [14]:
# ✅ STEP 5: 간단한 학습 데이터셋 정의
# 질문-답변 형태의 짧은 데이터셋을 정의합니다.
# 실습 목적이므로 간단한 예시로 구성되어 있습니다.
data = {
    "text": [
        "### 질문: joy강사의 별명은?\n### 답변: 스마일",
        "### 질문: 바다는 왜 파란가요?\n### 답변: 햇빛의 산란",
    ]
}
# Hugging Face Dataset 객체로 변환
dataset = Dataset.from_dict(data)


In [15]:
# ✅ STEP 6: 토크나이즈 함수
# [포인트] 텍스트 데이터를 모델에 넣기 위해 숫자 형태(토큰 ID)로 변환합니다.
# padding과 truncation을 설정해서 입력 길이를 일정하게 맞춰줍니다.

def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)

# 데이터셋에 토크나이즈 함수를 적용하여 숫자형 시퀀스로 변환
tokenized_dataset = dataset.map(tokenize_function)


Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [16]:
# ✅ STEP 7: 데이터 콜레이터
# [개념] 데이터 콜레이터는 배치로 묶을 때 패딩, 마스킹 등을 자동으로 처리해주는 도구입니다.
# GPT처럼 다음 단어를 예측하는 방식에서는 MLM(False)로 설정합니다.

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


데이터 콜레이터란?
DataCollatorForLanguageModeling은 모델에 데이터를 넣기 직전에 문장들을 자동으로 정리해주는 도구입니다.
모델이 문장을 한 번에 여러 개 처리하려면 길이를 맞춰야 하는데, 그 과정을 **패딩(padding)**이라고 해요.
또, 어떤 위치를 학습해야 할지 표시해주는 **마스킹(masking)**도 필요하죠.

이 과정을 직접 처리하지 않아도 되게끔, 데이터 콜레이터가 알아서 처리해 줍니다.

In [17]:
# ✅ STEP 8: 트레이닝 설정

# [포인트] 모델 학습에 필요한 하이퍼파라미터들을 설정합니다.
training_args = TrainingArguments(
    output_dir="./results",  # 결과 디렉토리
    per_device_train_batch_size=10,  # GPU나 CPU 1개당 배치 사이즈
    num_train_epochs=50,  # 데이터 전체를 몇 번 반복해서 학습할지 (여기선 10번)
    logging_steps=1,  # 1스텝마다 로그 출력 (학습 진행 상황 확인)
    save_strategy="no",  # 중간 저장 생략 (데모 목적이므로)
    fp16=False,  # GPU 없이 CPU로 학습 시 False 설정,
         report_to = "none",  # wandb, tensorboard 등 외부 로깅 툴 비활성화

)


✅ step과 epoch의 차이 정리

- **step은 iteration과 동일한 의미**입니다. 즉, 한 번의 파라미터 업데이트를 뜻합니다.
- 모델이 **한 배치(batch)**를 보고 학습하는 것이 1 step (1 iteration)입니다.
- 반면 **epoch**은 전체 데이터를 한 바퀴 학습한 것을 의미합니다.
- 예: 데이터 100개, 배치 크기 10이면 → 10 step = 1 epoch입니다.
- 정리하면, **여러 step이 모여 하나의 epoch을 구성**합니다.


In [18]:
# ✅ STEP 9: 트레이너 설정 및 학습 시작
# Hugging Face의 Trainer 클래스를 사용해 학습을 진행합니다.
# 위에서 정의한 모델, 데이터, 설정값을 모두 전달하여 학습을 시작합니다.\
import numpy as np
# 정확도를 계산하는 함수 정의
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # padding 부분은 무시
    mask = labels != -100
    correct = (predictions == labels) & mask
    accuracy = correct.sum() / mask.sum()

    return {"accuracy": accuracy}
trainer = Trainer(
    model=model,  # 학습할 모델
    args=training_args,  # 학습 설정
    train_dataset=tokenized_dataset,  # 학습 데이터
    # tokenizer=tokenizer,  # 텍스트 디코딩용   # Transformers 4.x 문법
    processing_class=tokenizer,  # 텍스트 디코딩용, Transformers 5.x 문법
    data_collator=data_collator,  # 배치 구성 도우미
    compute_metrics=compute_metrics  # <-- 여기 추가

)

# 학습 시작!
trainer.train()

Step,Training Loss
1,3.270241
2,3.545980
3,3.488761
4,3.404766
5,3.636006
6,3.433086
7,3.636255
8,3.382041
9,3.310548
10,3.602261


TrainOutput(global_step=50, training_loss=3.4298707246780396, metrics={'train_runtime': 2.8501, 'train_samples_per_second': 35.087, 'train_steps_per_second': 17.543, 'total_flos': 6656871628800.0, 'train_loss': 3.4298707246780396, 'epoch': 50.0})

In [19]:
# "우리집 강아지 이름은?"이라는 질문에 답을 하도록 구성합니다.
input_text = "### 질문:joy 강사의 별명은?\n### 답변:"

# 입력 문장을 숫자로 바꿔주는 tokenizer를 사용해 모델이 이해할 수 있는 형식으로 변환합니다.
# return_tensors="pt"는 PyTorch 텐서 형태로 반환하겠다는 뜻입니다.
inputs = tokenizer(input_text, return_tensors="pt")

# 모델이 올라가 있는 디바이스(GPU 또는 CPU)를 가져옵니다.
device = model.device

# 입력 데이터도 모델이 있는 디바이스로 옮겨줍니다.
# 그래야 모델과 데이터가 같은 장치에 있어 연산이 가능합니다.
inputs = {k: v.to(device) for k, v in inputs.items()}

# 모델에게 답변을 생성하도록 지시합니다.
# max_new_tokens=50은 최대 50개의 새로운 단어(토큰)를 생성하겠다는 의미입니다.
outputs = model.generate(**inputs, max_new_tokens=50)

# 생성된 답변을 사람이 읽을 수 있는 문자열로 바꿔서 출력합니다.
# skip_special_tokens=True는 시작/종료 같은 특수 기호는 출력하지 않겠다는 의미입니다.
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


### 질문:joy 강사의 별명은?
### 답변: 가지는 가아 가�는�� 가아아 가아아 �


# GPT2파인튜닝 Full FineTuning(lora 제거)

In [21]:
# STEP 1: 환경 설정
# uv add transformers datasets


In [24]:
# STEP 2: 기본 라이브러리 임포트
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

# STEP 3: 모델 및 토크나이저 로드
model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id)
model.resize_token_embeddings(len(tokenizer))  # pad_token 추가 고려

# STEP 4: 학습 데이터 정의
data = {
    "text": [
        "### 질문: 김정현강사의 별명은?\n### 답변: 유니코",
        "### 질문: 바다는 왜 파란가요?\n### 답변: 햇빛의 산란",
    ]
}
dataset = Dataset.from_dict(data)

# STEP 5: 토큰화
def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_function)

# STEP 6: 데이터 콜레이터
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# STEP 7: 트레이닝 설정
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=10,
    num_train_epochs=50,
    logging_steps=1,
    save_strategy="no",
    fp16=False,
         report_to = "none",  # wandb, tensorboard 등 외부 로깅 툴 비활성화

)

import numpy as np

# STEP 8: 트레이너 설정 및 학습


# 정확도를 계산하는 함수 정의
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # padding 부분은 무시
    mask = labels != -100
    correct = (predictions == labels) & mask
    accuracy = correct.sum() / mask.sum()

    return {"accuracy": accuracy}
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics # <-- 여기 추가

)

trainer.train()


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Step,Training Loss
1,3.552148
2,3.118419
3,2.643848
4,2.542405
5,2.160382
6,2.034612
7,1.775009
8,1.865962
9,1.653320
10,1.204589


TrainOutput(global_step=50, training_loss=0.7487830194830895, metrics={'train_runtime': 4.1505, 'train_samples_per_second': 24.093, 'train_steps_per_second': 12.047, 'total_flos': 6532300800000.0, 'train_loss': 0.7487830194830895, 'epoch': 50.0})

In [25]:

# STEP 9: 추론 예시
input_text = "### 질문: joy강사의 별명은?\n### 답변:"
inputs = tokenizer(input_text, return_tensors="pt")

# 모델과 같은 디바이스로 이동
device = model.device
inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


### 질문: joy강사의 별명은?
### 답변: 햇�유니코?
### 햇빛의 산란가요?
### �변: �
